In [ ]:
# ============================================================
# Regression Evaluation Metrics — California Housing Dataset
# معیارهای ارزیابی مدل رگرسیون
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
# ------------------------------------------------------------
# Load California Housing Dataset
# ------------------------------------------------------------
# Each row = one block group (small geographic area in California)
# Target  = MedHouseVal: median house value in $100,000 units
# Note    : values above $500k were capped at 5.0 (normalized)
# Source  : 1990 U.S. Census

data = fetch_california_housing()
df   = pd.DataFrame(data.data, columns=data.feature_names)
df['MedHouseVal'] = data.target

print(f"Dataset shape : {df.shape}")
print(f"Features      : {list(data.feature_names)}")
print(f"Target        : MedHouseVal (median house value in $100k)")
print(f"Missing values: {df.isnull().sum().sum()}")
df.head()

In [ ]:
# ------------------------------------------------------------
# Geographic map of house prices
# Color   : house price  (red = expensive, blue = cheap)
# Size    : block population
# The vertical stripe on the right is caused by the $500k cap —
# all houses priced above $500k were set to 5.0
# ------------------------------------------------------------

scatter = plt.scatter(
    x=df['Longitude'],       
      # longitude = east-west position
    y=df['Latitude'],         # latitude  = north-south position
    c=df['MedHouseVal'],      # color = median house value
    cmap='RdYlBu_r',          # red = expensive, blue = cheap
    s=df['Population'] / 100, # circle size ∝ population
    alpha=0.4
)
plt.colorbar(scatter, label='Median House Value ($100k)')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.title('California Housing Prices\n(red=expensive, blue=cheap, size=population)')
plt.tight_layout()
plt.show()

In [ ]:
# ------------------------------------------------------------
# Correlation heatmap
# Shows how strongly each feature is linearly related to others
# Key insight: MedInc has the strongest correlation with price (~0.69)
# ------------------------------------------------------------

cm = df.corr()
sns.heatmap(cm, annot=True, fmt='.2f', cmap='coolwarm', center=0, linewidths=0.5)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# ------------------------------------------------------------
# Train / Test Split  (جدا سازی داده‌ها)
# 80% → training   (model learns weights from this)
# 20% → testing    (held out to evaluate real-world performance)
# random_state=42  → guarantees the same split every run
# ------------------------------------------------------------

X = df.drop('MedHouseVal', axis=1)  # feature matrix
y = df['MedHouseVal']               # target vector

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")
print(f"y_train : {y_train.shape}")
print(f"y_test  : {y_test.shape}")

In [ ]:
# ------------------------------------------------------------
# Train Linear Regression model
# Formula : ŷ = w0 + w1*x1 + w2*x2 + ... + wn*xn
# The model minimizes Sum of Squared Residuals to find best weights
# ------------------------------------------------------------

model = LinearRegression()
model.fit(X_train, y_train)   # learn weights from training data

# Show learned coefficients (one per feature)
coef_df = pd.DataFrame({'Feature': X.columns, 'Coefficient': model.coef_})
coef_df = coef_df.sort_values('Coefficient', key=abs, ascending=False)

print(f"Intercept (w0) = {model.intercept_:.4f}")
print()
coef_df

In [ ]:
# ------------------------------------------------------------
# Generate predictions on the unseen test set
# y_pred[i] = model's estimated house value for test sample i
# ------------------------------------------------------------

y_pred = model.predict(X_test)

# Quick sanity check: first 10 actual vs predicted
check = pd.DataFrame({
    'Actual (y)'    : y_test.values[:10],
    'Predicted (ŷ)' : y_pred[:10],
    'Error (y - ŷ)' : y_test.values[:10] - y_pred[:10]
})
check.round(4)

In [ ]:
# ============================================================
# METRIC 1 — MAE  (Mean Absolute Error)
#            میانگین قدر مطلق خطا
# ------------------------------------------------------------
# Formula : MAE = (1/n) * Σ |y_i - ŷ_i|
# Steps   : 1) compute residuals  e_i = y_i - ŷ_i
#           2) take absolute value |e_i|
#           3) average them
# Pros    : same unit as target, robust to outliers
# Cons    : does not penalize large errors more than small ones
# ============================================================

errors          = y_test.values - y_pred       # step 1: residuals
absolute_errors = np.abs(errors)               # step 2: |e_i|
MAE             = np.mean(absolute_errors)     # step 3: mean

MAE_sklearn = mean_absolute_error(y_test, y_pred)  # verify with sklearn

print(f"MAE (manual)  = {MAE:.6f}")
print(f"MAE (sklearn) = {MAE_sklearn:.6f}")
print(f"→ On average, prediction is off by ${MAE * 100_000:,.0f}")

In [ ]:
# ============================================================
# METRIC 2 — MSE  (Mean Squared Error)
#            میانگین مجذور خطا
# ------------------------------------------------------------
# Formula : MSE = (1/n) * Σ (y_i - ŷ_i)²
# Steps   : 1) compute residuals  e_i = y_i - ŷ_i
#           2) square them        e_i²
#           3) average them
# Pros    : differentiable → used as loss function during training
#           penalizes large errors more heavily (squared)
# Cons    : unit is target² → hard to interpret directly
# ============================================================

squared_errors = errors ** 2               # step 2: e_i²
MSE            = np.mean(squared_errors)   # step 3: mean

MSE_sklearn = mean_squared_error(y_test, y_pred)  # verify with sklearn

print(f"MSE (manual)  = {MSE:.6f}")
print(f"MSE (sklearn) = {MSE_sklearn:.6f}")
print(f"Unit: ($100k)²  →  not directly interpretable as a dollar amount")

In [ ]:
# ============================================================
# METRIC 3 — RMSE  (Root Mean Squared Error)
#            ریشه میانگین مجذور خطا
# ------------------------------------------------------------
# Formula : RMSE = sqrt(MSE) = sqrt( (1/n) * Σ (y_i - ŷ_i)² )
# Steps   : take the square root of MSE to restore original unit
# Pros    : same unit as target, still penalizes large errors
# Cons    : more sensitive to outliers than MAE
# Note    : RMSE > MAE always → the gap shows how many large errors exist
# ============================================================

RMSE = np.sqrt(MSE)   # square root of MSE

print(f"RMSE          = {RMSE:.6f}")
print(f"MAE           = {MAE:.6f}")
print(f"→ Typical prediction error : ${RMSE * 100_000:,.0f}")
print(f"→ RMSE > MAE confirms large outlier errors exist in the data")

In [ ]:
# ============================================================
# METRIC 4 — R²  (Coefficient of Determination)
#            ضریب تعیین
# ------------------------------------------------------------
# Formula : R² = 1 - SS_res / SS_tot
#   SS_res = Σ (y_i - ŷ_i)²   ← what our model did NOT explain
#   SS_tot = Σ (y_i - ȳ)²     ← total variance in y
# Intuition: proportion of variance in y explained by the model
#   R² = 1.0  → perfect prediction
#   R² = 0.5  → model explains 50% of the variance
#   R² = 0.0  → model is no better than predicting the mean
#   R² < 0    → model is worse than predicting the mean
# Cons    : always increases when more features are added
#           → use Adjusted R² for model comparison
# ============================================================

y_mean = np.mean(y_test.values)                         # ȳ: mean of actual values
SS_res = np.sum((y_test.values - y_pred) ** 2)          # residual sum of squares
SS_tot = np.sum((y_test.values - y_mean) ** 2)          # total sum of squares

r2 = 1 - (SS_res / SS_tot)                             # R² formula

r2_sklearn = r2_score(y_test, y_pred)                  # verify with sklearn

print(f"SS_res        = {SS_res:.4f}")
print(f"SS_tot        = {SS_tot:.4f}")
print(f"R² (manual)   = {r2:.6f}")
print(f"R² (sklearn)  = {r2_sklearn:.6f}")
print(f"→ Model explains {r2 * 100:.1f}% of the variance in house prices")

In [ ]:
# ============================================================
# METRIC 5 — Adjusted R²  (R² تنظیم‌شده)
# ------------------------------------------------------------
# Formula : Adj_R² = 1 - (1 - R²) * (n - 1) / (n - p - 1)
#   n = number of test samples
#   p = number of features (predictors)
# Why    : regular R² always increases when features are added,
#          even useless ones. Adjusted R² penalizes for extra features.
#   new feature is useful   → Adj R² increases
#   new feature is useless  → Adj R² decreases or stays same
# Note   : no sklearn function → must compute manually
# ============================================================

n = len(y_test)          # number of test samples
p = X_test.shape[1]      # number of features

adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)

print(f"n (samples)   = {n}")
print(f"p (features)  = {p}")
print(f"R²            = {r2:.6f}")
print(f"Adjusted R²   = {adj_r2:.6f}")
print(f"Difference    = {r2 - adj_r2:.6f}  (small because n >> p)")

In [ ]:
# ============================================================
# METRIC 6 — MAPE  (Mean Absolute Percentage Error)
#            میانگین درصد قدر مطلق خطا
# ------------------------------------------------------------
# Formula : MAPE = (1/n) * Σ | (y_i - ŷ_i) / y_i | * 100
# Intuition: error as a percentage of the actual value
# Pros    : scale-independent, easy to explain to non-technical stakeholders
# Cons    : undefined when y_i = 0, biased when actuals are very small
# ============================================================

mask            = y_test.values != 0                               # avoid division by zero
y_safe          = y_test.values[mask]
pred_safe       = y_pred[mask]

MAPE = np.mean(np.abs((y_safe - pred_safe) / y_safe)) * 100       # percentage

print(f"MAPE          = {MAPE:.4f}%")
print(f"→ On average, predictions are off by {MAPE:.1f}% of the actual house value")

In [ ]:
# ------------------------------------------------------------
# Summary: all metrics in one table
# ------------------------------------------------------------

summary = pd.DataFrame({
    'Metric': ['MAE', 'MSE', 'RMSE', 'R²', 'Adjusted R²', 'MAPE (%)'],
    'Value' : [MAE, MSE, RMSE, r2, adj_r2, MAPE],
    'Unit'  : ['$100k', '($100k)²', '$100k', 'unitless', 'unitless', '%'],
    'Interpretation': [
        f'avg error ≈ ${MAE * 100_000:,.0f}',
        'penalizes large errors (squared)',
        f'typical error ≈ ${RMSE * 100_000:,.0f}',
        f'{r2 * 100:.1f}% variance explained',
        f'{adj_r2 * 100:.1f}% variance explained (corrected for {p} features)',
        f'off by {MAPE:.1f}% of actual value'
    ]
})
summary.set_index('Metric', inplace=True)
summary['Value'] = summary['Value'].round(4)
summary

In [ ]:
# ------------------------------------------------------------
# Plot 1: Actual vs Predicted  +  Residuals distribution
# ------------------------------------------------------------
# Left  : every point is one test sample
#         red dashed line = perfect prediction (y == ŷ)
#         points close to line → small error
# Right : histogram of residuals (y - ŷ)
#         good model → centered at 0, bell-shaped
# ------------------------------------------------------------

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- actual vs predicted ---
axes[0].scatter(y_test, y_pred, alpha=0.2, s=5, color='steelblue')
lo, hi = min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())
axes[0].plot([lo, hi], [lo, hi], 'r--', lw=2, label='Perfect prediction')
axes[0].set_xlabel('Actual (y)')
axes[0].set_ylabel('Predicted (ŷ)')
axes[0].set_title(f'Actual vs Predicted  |  R² = {r2:.4f}')
axes[0].legend()

# --- residuals histogram ---
residuals = y_test.values - y_pred
axes[1].hist(residuals, bins=80, color='coral', edgecolor='white', alpha=0.8)
axes[1].axvline(0,               color='black', lw=2, linestyle='--', label='Zero error')
axes[1].axvline(residuals.mean(), color='blue',  lw=2, label=f'Mean = {residuals.mean():.3f}')
axes[1].set_xlabel('Residual (y − ŷ)')
axes[1].set_ylabel('Count')
axes[1].set_title('Residuals Distribution')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ------------------------------------------------------------
# Plot 2: MAE vs RMSE bar chart
# RMSE > MAE always
# The larger the gap, the more large outlier errors the model makes
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(7, 5))

bars = ax.bar(['MAE', 'RMSE'], [MAE, RMSE], color=['#3498db', '#e74c3c'], width=0.4)

for bar, val in zip(bars, [MAE, RMSE]):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.005,
            f'{val:.4f}', ha='center', fontsize=12, fontweight='bold')

ax.set_ylabel('Error ($100k)')
ax.set_title('MAE vs RMSE\n(both in same unit — $100k)')
ax.set_ylim(0, max(MAE, RMSE) * 1.25)
plt.tight_layout()
plt.show()

In [ ]:
# ------------------------------------------------------------
# Plot 3: Understanding R²
# Left  : baseline model → always predict the mean (ȳ)
# Right : our model      → predicts actual ŷ values
# The reduction in total error is what R² measures
# ------------------------------------------------------------

idx         = np.random.choice(len(y_test), 200, replace=False)
y_s         = y_test.values[idx]
ypred_s     = y_pred[idx]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# baseline: predict the mean
axes[0].scatter(range(200), y_s, s=10, alpha=0.5, label='Actual y')
axes[0].axhline(y_mean, color='red', lw=2, label=f'Baseline ȳ = {y_mean:.2f}')
for i in range(200):
    axes[0].plot([i, i], [y_s[i], y_mean], 'r-', alpha=0.05)   # vertical error lines
axes[0].set_title(f'Baseline (always predict mean)\nSS_tot = {SS_tot:.1f}')
axes[0].legend()
axes[0].set_ylim(0, 6)

# our model
axes[1].scatter(range(200), y_s,      s=10, alpha=0.5,               label='Actual y')
axes[1].scatter(range(200), ypred_s,  s=10, alpha=0.5, color='green', label='Predicted ŷ')
for i in range(200):
    axes[1].plot([i, i], [y_s[i], ypred_s[i]], 'g-', alpha=0.05)   # residual lines
axes[1].set_title(f'Our Model (Linear Regression)\nSS_res = {SS_res:.1f}  |  R² = {r2:.3f}')
axes[1].legend()
axes[1].set_ylim(0, 6)

fig.suptitle('R² = 1 - SS_res / SS_tot  (how much better are we than predicting the mean?)', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ------------------------------------------------------------
# When to use which metric?
# ------------------------------------------------------------

guide = pd.DataFrame({
    'Metric'     : ['MAE', 'MSE', 'RMSE', 'R²', 'Adjusted R²', 'MAPE'],
    'Use when'   : [
        'Outliers exist — you want all errors weighted equally',
        'Training / optimization (differentiable loss function)',
        'Large errors are unacceptable (e.g. medical, finance)',
        'Comparing models on the same dataset',
        'Choosing how many features to include in the model',
        'Comparing models across different datasets / scales'
    ]
})
guide.set_index('Metric', inplace=True)
print(guide.to_string())